# W1D2 실험: 색공간별 Density Plot 비교

> 📖 원리 이해: [LearnOpenCV - Color Spaces in OpenCV](https://learnopencv.com/color-spaces-in-opencv-cpp-python/)
> 📋 파라미터 확인: [OpenCV 공식 - cvtColor](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html)

## 📌 오늘의 목표
원본 튜토리얼의 핵심 실험을 재현합니다:
1. 같은 색의 물체를 **여러 조명 조건**에서 촬영 (시뮬레이션)
2. 각 이미지에서 해당 색 영역을 **크롭**
3. 모든 픽셀을 합쳐서 **4가지 색공간(RGB, HSV, LAB, YCrCb)** 의 채널별 **density plot** 비교
4. density plot에서 **threshold 범위를 읽어서** `inRange()`로 색 검출

## 🎯 배울 함수
| 함수 | 설명 |
|------|------|
| `cv2.cvtColor()` | 색공간 변환 |
| `cv2.split()` | 다채널 이미지를 채널별로 분리 |
| `cv2.inRange()` | 특정 범위의 픽셀만 마스크로 추출 |
| `cv2.bitwise_and()` | 마스크 적용 |

## 📦 import + 조명 시뮬레이션 이미지 생성

원본 튜토리얼에서는 큐브 이미지 10장을 다른 조명에서 촬영합니다.
우리는 같은 효과를 numpy로 시뮬레이션합니다.

**테스트 이미지**: 오렌지, 초록, 파랑 3가지 색 블록 + 회색 배경
**조명 시뮬레이션**: 기본 / 어두운 / 밝은 / 따뜻한 / 차가운 / 어둡고 따뜻한 — 총 6가지

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

PATCH_SIZE = 80
np.random.seed(42)

base_colors = {
    "orange": np.array([230, 120, 30], dtype=np.float64),
    "green":  np.array([40, 180, 60], dtype=np.float64),
    "blue":   np.array([30, 70, 210], dtype=np.float64),
}

lighting_conditions = {
    "기본":       {"brightness": 1.0,  "color_shift": (0, 0, 0)},
    "어두운":     {"brightness": 0.6,  "color_shift": (0, 0, 0)},
    "밝은":       {"brightness": 1.35, "color_shift": (0, 0, 0)},
    "따뜻한":     {"brightness": 1.0,  "color_shift": (25, 5, -20)},
    "차가운":     {"brightness": 1.0,  "color_shift": (-25, -5, 25)},
    "어둡고따뜻": {"brightness": 0.7,  "color_shift": (15, 5, -10)},
}

def make_patch(base_color, brightness=1.0, color_shift=(0, 0, 0), noise_std=8):
    patch = np.zeros((PATCH_SIZE, PATCH_SIZE, 3), dtype=np.float64)
    shifted = base_color * brightness + np.array(color_shift, dtype=np.float64)
    patch[:] = shifted
    noise = np.random.randn(PATCH_SIZE, PATCH_SIZE, 3) * noise_std
    patch += noise
    return np.clip(patch, 0, 255).astype(np.uint8)

def make_test_image(brightness=1.0, color_shift=(0, 0, 0)):
    bg = np.full((PATCH_SIZE * 2, PATCH_SIZE * 3, 3), 128, dtype=np.float64)
    bg = bg * brightness + np.array(color_shift)
    bg += np.random.randn(*bg.shape) * 5
    img = np.clip(bg, 0, 255).astype(np.uint8)
    margin = 10
    s = PATCH_SIZE - margin * 2
    for i, (name, base) in enumerate(base_colors.items()):
        patch = make_patch(base, brightness, color_shift)
        y1, x1 = margin + PATCH_SIZE//4, margin + i * PATCH_SIZE
        img[y1:y1+s, x1:x1+s] = patch[margin:margin+s, margin:margin+s]
    return img

test_images = {}
for name, params in lighting_conditions.items():
    test_images[name] = make_test_image(**params)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, (name, img) in enumerate(test_images.items()):
    ax = axes[i // 3, i % 3]
    ax.imshow(img)
    ax.set_title(f"{name} 조명", fontsize=13, fontweight='bold')
    ax.axis('off')

plt.suptitle("같은 색 블록 — 6가지 조명 조건", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 1단계: 각 이미지에서 오렌지 영역 크롭 + 픽셀 수집

원본 튜토리얼에서 큐브의 각 색 면을 크롭하는 것처럼,
6장의 이미지에서 오렌지 블록 위치를 잘라서 모든 픽셀을 합칩니다.

이렇게 하면 **"같은 색인데 조명 때문에 RGB 값이 다른"** 픽셀 데이터가 만들어집니다.

In [ ]:
margin = 10
s = PATCH_SIZE - margin * 2
y1 = margin + PATCH_SIZE // 4
x1 = margin

all_orange_pixels = []
fig, axes = plt.subplots(1, len(test_images), figsize=(20, 3))

for i, (name, img) in enumerate(test_images.items()):
    crop = img[y1:y1+s, x1:x1+s]
    all_orange_pixels.append(crop.reshape(-1, 3))
    
    axes[i].imshow(crop)
    mean_rgb = crop.mean(axis=(0, 1))
    axes[i].set_title(f"{name}\nR={mean_rgb[0]:.0f} G={mean_rgb[1]:.0f} B={mean_rgb[2]:.0f}", fontsize=10)
    axes[i].axis('off')

all_orange_rgb = np.vstack(all_orange_pixels)
plt.suptitle(f"6가지 조명에서 크롭한 오렌지 영역 (총 {len(all_orange_rgb):,}픽셀)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"각 크롭: {s}x{s} = {s**2:,}픽셀")
print(f"전체 합계: {len(all_orange_rgb):,}픽셀")

## 🔧 2단계: 4가지 색공간으로 변환

수집한 오렌지 픽셀을 RGB, HSV, LAB, YCrCb로 변환합니다.

| 색공간 | 밝기 채널 | 색 채널 |
|--------|-----------|--------|
| RGB | 없음 (3채널 모두 밝기+색 혼합) | R, G, B |
| HSV | V (명도) | H (색상), S (채도) |
| LAB | L (밝기) | A (녹↔자홍), B (청↔황) |
| YCrCb | Y (밝기) | Cr (적색차), Cb (청색차) |

In [ ]:
pixels_as_img = all_orange_rgb.reshape(1, -1, 3).astype(np.uint8)
pixels_bgr = cv2.cvtColor(pixels_as_img, cv2.COLOR_RGB2BGR)
pixels_hsv = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2HSV).reshape(-1, 3)
pixels_lab = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2LAB).reshape(-1, 3)
pixels_ycrcb = cv2.cvtColor(pixels_bgr, cv2.COLOR_BGR2YCrCb).reshape(-1, 3)
pixels_rgb = all_orange_rgb

color_spaces = {
    "RGB":   (pixels_rgb,   ["R", "G", "B"],  ["red", "green", "blue"]),
    "HSV":   (pixels_hsv,   ["H", "S", "V"],  ["orange", "cyan", "gray"]),
    "LAB":   (pixels_lab,   ["L", "A", "B"],  ["gray", "magenta", "gold"]),
    "YCrCb": (pixels_ycrcb, ["Y", "Cr", "Cb"],["gray", "red", "blue"]),
}

## 🔧 3단계: Density Plot (채널별 히스토그램)

원본 튜토리얼의 핵심 시각화입니다.
같은 "오렌지색" 픽셀이 각 색공간의 각 채널에서 어떤 분포를 보이는지 봅니다.

- **분포가 좁다** = threshold 범위를 좁게 잡아도 된다 = 색 검출이 쉽다
- **분포가 넓다** = threshold를 넓게 잡아야 해서 다른 색도 섞일 위험이 있다

In [ ]:
fig, axes = plt.subplots(4, 3, figsize=(18, 16))

for row, (cs_name, (pixels, ch_names, ch_colors)) in enumerate(color_spaces.items()):
    for col in range(3):
        ax = axes[row, col]
        ch_data = pixels[:, col]
        
        hist_range = [0, 180] if (cs_name == "HSV" and col == 0) else [0, 256]
        n_bins = 180 if (cs_name == "HSV" and col == 0) else 256
        
        ax.hist(ch_data, bins=n_bins, range=hist_range,
                color=ch_colors[col], alpha=0.7, edgecolor='none', density=True)
        
        mn, mx = ch_data.min(), ch_data.max()
        ax.axvline(mn, color='black', linewidth=1, linestyle=':')
        ax.axvline(mx, color='black', linewidth=1, linestyle=':')
        
        spread = mx - mn
        ax.set_title(f"{cs_name} - {ch_names[col]}\nrange: [{mn} ~ {mx}] (폭={spread})",
                     fontsize=12, fontweight='bold')
        
        if row == 3:
            ax.set_xlabel("Pixel Value", fontsize=10)
        if col == 0:
            ax.set_ylabel("Density", fontsize=10)

plt.suptitle("Density Plot: 같은 오렌지색의 채널별 분포 — 색공간 비교\n"
             "(분포가 좁을수록 threshold 잡기 쉬움)",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드

**여기서 봐야 할 것**: 각 채널의 분포 **폭(range)**입니다.

- **RGB**: R, G, B 세 채널 모두 분포가 넓게 퍼짐 → 조명이 바뀌면 세 값이 동시에 변하기 때문
- **HSV**: **H(색상) 채널의 폭이 매우 좁음** → 조명이 바뀌어도 색상 자체는 유지됨. V(밝기)만 변함
- **LAB**: L은 넓지만 A, B는 비교적 좁음 → 밝기와 색이 분리되어 있기 때문
- **YCrCb**: Y는 넓지만 Cr, Cb는 비교적 좁음 → LAB과 비슷한 원리

> 🤔 **판단 질문:** RGB의 R 채널 폭과 HSV의 H 채널 폭을 비교해보세요. 어느 쪽이 threshold 잡기 쉬운가요?

## 🔧 4단계: Density Plot에서 threshold 읽기 → inRange() 검출

원본 튜토리얼의 방법:
- density plot에서 분포가 **가장 밀집된 범위**를 읽어서 `inRange()`의 lower/upper로 사용
- 우리는 각 채널의 **[min, max]** 범위를 그대로 threshold로 사용합니다

이 threshold를 테스트 이미지(3가지 색 + 배경)에 적용해서,
**오렌지만 깔끔하게 분리되는지** 비교합니다.

In [ ]:
def get_threshold_from_density(pixels):
    lower = pixels.min(axis=0).astype(np.uint8)
    upper = pixels.max(axis=0).astype(np.uint8)
    return lower, upper

rgb_lower, rgb_upper = get_threshold_from_density(pixels_rgb)
hsv_lower, hsv_upper = get_threshold_from_density(pixels_hsv)
lab_lower, lab_upper = get_threshold_from_density(pixels_lab)
ycrcb_lower, ycrcb_upper = get_threshold_from_density(pixels_ycrcb)

print("Density plot에서 읽은 threshold 범위:")
print(f"  RGB:   lower={rgb_lower}, upper={rgb_upper}")
print(f"  HSV:   lower={hsv_lower}, upper={hsv_upper}")
print(f"  LAB:   lower={lab_lower}, upper={lab_upper}")
print(f"  YCrCb: lower={ycrcb_lower}, upper={ycrcb_upper}")

test_img = test_images["기본"]
test_bgr = cv2.cvtColor(test_img, cv2.COLOR_RGB2BGR)

masks = {
    "RGB":   cv2.inRange(test_img, rgb_lower, rgb_upper),
    "HSV":   cv2.inRange(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2HSV), hsv_lower, hsv_upper),
    "LAB":   cv2.inRange(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2LAB), lab_lower, lab_upper),
    "YCrCb": cv2.inRange(cv2.cvtColor(test_bgr, cv2.COLOR_BGR2YCrCb), ycrcb_lower, ycrcb_upper),
}

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
fig.suptitle("inRange() 검출 결과 — 기본 조명 이미지", fontsize=16, fontweight='bold')

for col, (cs_name, mask) in enumerate(masks.items()):
    axes[0, col].imshow(test_img)
    axes[0, col].set_title(f"원본", fontsize=12)
    axes[0, col].axis('off')
    
    axes[1, col].imshow(mask, cmap='gray')
    axes[1, col].set_title(f"{cs_name} 마스크", fontsize=12, fontweight='bold')
    axes[1, col].axis('off')
    
    result = cv2.bitwise_and(test_img, test_img, mask=mask)
    axes[2, col].imshow(result)
    axes[2, col].set_title(f"{cs_name} 결과", fontsize=12)
    axes[2, col].axis('off')

plt.tight_layout()
plt.show()

### 해석 가이드

- **RGB 마스크**: 오렌지 외에 다른 색이나 배경도 같이 잡힐 수 있음 (밝기+색 혼합이라 범위가 넓으니까)
- **HSV 마스크**: H 채널 범위가 좁아서 오렌지만 정확히 분리될 가능성이 높음
- **LAB/YCrCb 마스크**: HSV보다는 넓지만 RGB보다는 깔끔

## 🔧 5단계: 같은 threshold로 다른 조명의 이미지에도 적용

원본 튜토리얼의 핵심 실험입니다.
**기본 조명에서 만든 threshold를 다른 조명 이미지에 그대로 적용하면?**

- RGB: 조명이 바뀌면 threshold가 안 맞아서 검출 실패
- HSV/LAB/YCrCb: 밝기가 분리되어 있으므로 조명이 바뀌어도 색 채널은 비슷 → 검출 유지

In [ ]:
test_names = ["기본", "어두운", "밝은", "따뜻한", "차가운"]

fig, axes = plt.subplots(len(test_names), 5, figsize=(22, len(test_names) * 4))

for row, img_name in enumerate(test_names):
    img = test_images[img_name]
    img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"{img_name} 조명" if row == 0 else f"{img_name}", fontsize=12)
    axes[row, 0].set_ylabel(f"{img_name}", fontsize=13, fontweight='bold', rotation=0, labelpad=60)
    axes[row, 0].axis('off')
    
    cs_configs = [
        ("RGB",   img,                                          rgb_lower, rgb_upper),
        ("HSV",   cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV),    hsv_lower, hsv_upper),
        ("LAB",   cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB),    lab_lower, lab_upper),
        ("YCrCb", cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb), ycrcb_lower, ycrcb_upper),
    ]
    
    for col_offset, (cs_name, converted, lower, upper) in enumerate(cs_configs):
        mask = cv2.inRange(converted, lower, upper)
        result = cv2.bitwise_and(img, img, mask=mask)
        axes[row, col_offset + 1].imshow(result)
        if row == 0:
            axes[row, col_offset + 1].set_title(f"{cs_name}", fontsize=13, fontweight='bold')
        axes[row, col_offset + 1].axis('off')

plt.suptitle("같은 threshold를 다른 조명 이미지에 적용 — 어떤 색공간이 조명 변화에 강한가?",
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 해석 가이드

- **RGB**: 조명이 바뀌면 R, G, B 값이 동시에 바뀌므로 threshold 범위 밖으로 나감 → 검출 실패
- **HSV**: 조명이 바뀌면 V(밝기)만 바뀌고 H(색상)는 유지됨 → 검출 유지
- **LAB**: L(밝기)만 바뀌고 A, B(색)는 비슷 → 비교적 안정적
- **YCrCb**: Y(밝기)만 바뀌고 Cr, Cb(색)는 비슷 → 비교적 안정적

> 🤔 **판단 질문:** "어두운" 조명에서 RGB와 HSV 결과를 비교해보세요. 왜 이런 차이가 나나요?

## 📝 정리

### 오늘 배운 것
| 함수 | 핵심 포인트 |
|------|------------|
| `cv2.cvtColor()` | BGR→HSV, BGR→LAB, BGR→YCrCb 변환 |
| `cv2.split()` | 다채널 이미지를 채널별로 분리해서 density plot 그리기 |
| `cv2.inRange()` | density plot에서 읽은 범위로 마스크 생성 |
| `cv2.bitwise_and()` | 마스크를 원본에 적용해서 해당 색만 추출 |

### 핵심 결론 (= 원본 튜토리얼의 메시지)
- **RGB**: 밝기와 색이 섞여 있어서 density plot이 넓게 퍼짐 → threshold 잡기 어려움, 조명 변화에 취약
- **HSV**: H 채널 하나로 색상 특정 가능 → **가장 컴팩트, 첫 번째 선택지**
- **LAB**: A, B 채널이 컴팩트 → **두 번째 선택지** (HSV보다 약간 넓지만 RGB보다 훨씬 좋음)
- **YCrCb**: Cr, Cb 채널이 컴팩트 → **세 번째 선택지**

### 복습 퀴즈
1. RGB에서 조명이 밝아지면 R, G, B 세 값이 동시에 올라가는 이유는?
2. HSV에서 조명이 바뀌어도 H값이 유지되는 이유는? (힌트: V가 담당하는 것)
3. `cv2.inRange(img_hsv, (10,100,100), (25,255,255))` — 이 범위가 의미하는 것은?
4. 공장에서 AOI 카메라의 조명이 약간 바뀌었을 때, RGB로 검사하면 어떤 문제가 생기나?